# Notebook 04 — A Home for Your Vectors (ChromaDB)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

In Notebook 03 we found the closest sentence by comparing a query against **all twelve**
sentences by hand. Twelve is fine. A million is not — you don't want to write that loop, and
you don't want it slow. A **vector store** is a tool whose whole job is: hold a pile of
vectors, and instantly find the ones nearest a query.

> **Analogy:** a library with a magic catalogue. You describe what you want; it walks you
> straight to the closest shelf — without you scanning every book.

We'll use **ChromaDB**: free, local, nothing leaves your machine.

In [ ]:
%pip install -q sentence-transformers chromadb
print("Ready.")

## A quick primer on ChromaDB

A few words first, so the code reads clearly:

- A **database** is just an organised store of information you can put things into and get
  back out of.
- A **vector store** (or **vector database**) is a database built for *vectors*
  (embeddings). Its special skill is finding the stored vectors **nearest** to a query —
  the meaning-search from Notebook 01, done for you, and fast even with millions of items.
- A **collection** is one named group of items inside the store — like a single table, or
  one labelled drawer. You make one with `client.create_collection(name=...)`.
- The **documents** are the pieces of text you store. ChromaDB embeds each one for you
  (with MiniLM), so you hand it plain text, not numbers.
- Each item needs a unique **id** (a name tag like `"s0"`, `"s1"`) so the store can tell
  items apart.
- **metadata** is optional extra information you attach to an item (later we tuck an
  answer next to a question this way).
- A **query** is how you search: you give `collection.query(...)` some text, and it hands
  back the nearest stored documents.
- **distance / space**: we set the store to `"cosine"` space, so "closeness" means the
  cosine similarity you already know. ChromaDB reports a **distance**, and
  `similarity = 1 - distance`.

That is the whole vocabulary. Now let's build one.

## Step 1 — Create a store that speaks MiniLM

We tell ChromaDB to use the **same** MiniLM model from Notebook 03 to turn text into
vectors. Then every sentence we add is embedded automatically — we just hand it text. We also
set the store to measure closeness by **cosine** (angle), the same "pointing the same way"
notion we've used all along.

**What to expect when you run it:** the model downloads once (a short pause the first time),
then a line prints saying an empty collection has been created in the `./chroma_store` folder.
Nothing is searched yet; we have only built the empty drawer.

In [2]:
import chromadb                                       # the local vector store
from chromadb.utils import embedding_functions       # helpers that turn text into vectors

# Wrap the MiniLM model so ChromaDB can call it to embed any text we add or query.
minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"                    # same small model used in Notebook 03
)

# A store saved to a folder on disk, so it survives after the notebook closes.
client = chromadb.PersistentClient(path="./chroma_store")   # path = the folder it writes to

# Start fresh each run so re-running doesn't pile up duplicates.
try:
    client.delete_collection("sentences")            # remove the old drawer if it exists
except Exception:
    pass                                             # first run: nothing to delete, ignore

collection = client.create_collection(
    name="sentences",                                # the label for this group of items
    embedding_function=minilm_ef,                    # MiniLM embeds text for us automatically
    metadata={"hnsw:space": "cosine"},               # measure closeness by cosine angle
)
print("Empty collection created (it lives in the ./chroma_store folder).")

/Users/riteshmodi/gits/leanpub_courses/courses/vectors-and-embeddings/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14337.84it/s]

Empty collection created (it lives in the ./chroma_store folder).


## Step 2 — Add the twelve sentences

We hand ChromaDB plain text plus an id for each. It embeds them with MiniLM behind the
scenes and stores the vectors. No manual `encode` call needed.

**What to expect when you run it:** a single line reports `Stored 12 sentences.` The work of
turning each sentence into a vector happens silently inside `add`.

In [3]:
sentences = [
    "The dog wagged its tail when its owner came home.",
    "A kitten chased a ball of yarn across the floor.",
    "Lions live in prides on the African savanna.",
    "The parrot mimicked every word the children said.",
    "I drove the car to the grocery store this morning.",
    "The new electric bicycle has a range of sixty miles.",
    "Trucks deliver packages to our neighbourhood every day.",
    "The airplane landed smoothly despite the strong winds.",
    "She baked sourdough bread for the first time on Sunday.",
    "The pizza was hot and covered in melted mozzarella.",
    "I ordered sushi for lunch at the new Japanese restaurant.",
    "He grilled steak and roasted vegetables for dinner.",
]
# One unique id per sentence, like name tags: "sent_0", "sent_1", and so on.
ids = [f"sent_{i}" for i in range(len(sentences))]

# Hand over text and ids; ChromaDB embeds each sentence with MiniLM and stores the vectors.
collection.add(ids=ids, documents=sentences)
print(f"Stored {collection.count()} sentences.")     # count() = how many items are stored

Stored 12 sentences.


## Step 3 — Ask it a question

`collection.query` embeds your query, finds the nearest stored vectors, and hands back the
matching sentences — the same nearest-neighbour search from Notebook 01, now done for us at
any scale.

**What to expect when you run it:** the three **food** sentences come back first, even though
the query "something tasty to eat" never names pizza or sushi. The store returns a
**distance** (smaller means closer), and we convert it with `similarity = 1 - distance`. The
top match prints a similarity around 0.441.

In [4]:
result = collection.query(
    query_texts=["something tasty to eat"],          # the search phrase; ChromaDB embeds it
    n_results=3,                                     # ask for the 3 nearest sentences
)

print("Query: 'something tasty to eat'\n")
# query() can take several searches at once, so results are nested per query.
# We sent one query, so the [0] below picks the results for that single query.
for doc, dist in zip(result["documents"][0], result["distances"][0]):
    # With cosine space, similarity = 1 - distance. Higher = closer in meaning.
    print(f"  similarity {1 - dist:.3f}   {doc}")

Query: 'something tasty to eat'

  similarity 0.441   He grilled steak and roasted vegetables for dinner.
  similarity 0.340   I ordered sushi for lunch at the new Japanese restaurant.
  similarity 0.245   The pizza was hot and covered in melted mozzarella.


### What to notice
- The top results are the **food** sentences — even though the query never says "pizza" or
  "sushi". Meaning, not keywords.
- ChromaDB returns a **distance**; with cosine space, `similarity = 1 - distance`. Bigger
  similarity = closer in meaning.
- The store sits in the `./chroma_store` folder. Close the notebook, reopen, and the data is
  still there — no need to re-embed.

**Try it:** change the query to "a wild animal" or "ways to get around town" and re-run.

## Recap

- A vector store holds many vectors and finds the nearest ones to a query — fast, at any
  scale.
- ChromaDB embeds text for us (with MiniLM), stores it locally, and searches it.
- `add` puts text in; `query` pulls the closest text out. That's the whole interface.
- Cosine distance and cosine similarity are two sides of one coin: `sim = 1 - dist`.

**Next (Notebook 05):** we wire this into a tiny **semantic search engine** over a small
document collection — the thing promised in Lesson 0.

## Practice — Your Turn

Three short exercises using the `collection` you already built above. Read each task,
make your prediction where asked, then run the answer cell to check. Each answer cell
reuses the existing store, so run the notebook top to bottom first.

### Exercise 1 — Search with words the store has never seen

Search the collection for `"a furry pet that purrs"`. Notice that none of the twelve
stored sentences contain the words *furry*, *purrs*, or even *cat*. The store matches on
meaning, not on shared words.

Before you run the answer, predict which group of sentences will come back on top: the
animal ones, the vehicle ones, or the food ones.

Try it yourself, then run the answer cell below.

In [5]:
# Answer
# Search with a phrase that shares no words with any stored sentence.
result = collection.query(
    query_texts=["a furry pet that purrs"],   # no word here appears in the 12 sentences
    n_results=3,                              # ask for the 3 nearest sentences
)

print("Query: 'a furry pet that purrs'\n")
# We sent one query, so [0] picks the results for that single query.
for doc, dist in zip(result["documents"][0], result["distances"][0]):
    # Cosine space: similarity = 1 - distance. Higher means closer in meaning.
    print(f"  similarity {1 - dist:.3f}   {doc}")

# The kitten and other animal sentences rise to the top, matched by meaning alone.

Query: 'a furry pet that purrs'

  similarity 0.420   The dog wagged its tail when its owner came home.
  similarity 0.338   A kitten chased a ball of yarn across the floor.
  similarity 0.297   The parrot mimicked every word the children said.


### Exercise 2 — Turn a distance into a similarity by hand

ChromaDB hands back a **distance** for each match. With cosine space, you get the
similarity with one subtraction: `similarity = 1 - distance`. A smaller distance means
the two vectors point more nearly the same way, so they are closer in meaning.

Predict this first: if one match has distance 0.20 and another has distance 0.55, which
one is closer in meaning?

Try it yourself, then run the answer cell below.

In [6]:
# Answer
# Run a query and take the raw distance of the top match straight from the store.
result = collection.query(query_texts=["a wild animal"], n_results=3)

top_doc = result["documents"][0][0]      # the nearest sentence
top_dist = result["distances"][0][0]     # its distance (smaller = closer)

# Convert that distance to a similarity by hand: similarity = 1 - distance.
top_sim = 1 - top_dist

print(f"Top sentence: {top_doc}")
print(f"Distance returned by the store: {top_dist:.3f}")
print(f"Similarity (1 - distance):      {top_sim:.3f}")
print()

# Show the rule across all three results so the pattern is clear.
for doc, dist in zip(result["documents"][0], result["distances"][0]):
    print(f"  distance {dist:.3f}  ->  similarity {1 - dist:.3f}   {doc}")

# Smaller distance gives larger similarity, so the smaller-distance match is the closer one.

Top sentence: The dog wagged its tail when its owner came home.
Distance returned by the store: 0.588
Similarity (1 - distance):      0.412

  distance 0.588  ->  similarity 0.412   The dog wagged its tail when its owner came home.
  distance 0.697  ->  similarity 0.303   Lions live in prides on the African savanna.
  distance 0.716  ->  similarity 0.284   A kitten chased a ball of yarn across the floor.


### Exercise 3 — Add two new sentences and make one the top match

The store is not frozen. You can add more sentences any time with
`collection.add(ids=[...], documents=[...])`. Each new item needs a fresh id that is not
already in use. The twelve sentences use ids `"sent_0"` through `"sent_11"`, so we will
use brand-new ids `"sent_12"` and `"sent_13"` to avoid a clash.

We will add two sentences about gardening, then search for `"planting flowers in the
garden"`. Predict whether one of your two new sentences will come back as the top match.

Try it yourself, then run the answer cell below.

In [7]:
# Answer
# Two brand-new sentences to add to the store.
new_sentences = [
    "She planted tomatoes and roses in the back garden.",   # close to a garden query
    "The gardener watered the flower beds at dawn.",        # also about gardening
]
new_ids = ["sent_12", "sent_13"]   # fresh ids, not used by the original twelve

# Add only if these ids are not already present, so re-running the cell is safe.
existing = set(collection.get()["ids"])          # all ids currently stored
to_add = [i for i in new_ids if i not in existing]   # keep just the missing ones
if to_add:
    docs_to_add = [new_sentences[new_ids.index(i)] for i in to_add]  # matching texts
    collection.add(ids=to_add, documents=docs_to_add)  # ChromaDB embeds and stores them
    print(f"Added {len(to_add)} new sentence(s). Store now holds {collection.count()}.")
else:
    print(f"Already added. Store holds {collection.count()} sentences.")
print()

# Now search for something that matches one of the new gardening sentences.
result = collection.query(query_texts=["planting flowers in the garden"], n_results=3)

print("Query: 'planting flowers in the garden'\n")
for doc, dist in zip(result["documents"][0], result["distances"][0]):
    print(f"  similarity {1 - dist:.3f}   {doc}")

# One of the two new gardening sentences comes back on top, because it is closest in meaning.

Added 2 new sentence(s). Store now holds 14.

Query: 'planting flowers in the garden'

  similarity 0.622   She planted tomatoes and roses in the back garden.
  similarity 0.586   The gardener watered the flower beds at dawn.
  similarity 0.141   He grilled steak and roasted vegetables for dinner.
